四元数乘法中的左乘和右乘决定了旋转的参照系：
左乘（pre-multiply）：意味着你从全局坐标系（惯性系）出发旋转。
右乘（post-multiply）：意味着旋转是在局部坐标系（物体自身的坐标系）进行的。

In [ ]:
   ######################################
    ### Initialize Kalman Gain Network ###
    ######################################
        
def InitKGainNet(self, prior_Q, prior_Sigma, prior_S, args):
        """
            初始化 Kalman 增益网络，使用多个 GRU 和全连接网络对 Kalman 滤波过程中的关键变量进行建模。
            参数：
            - prior_Q：预测误差协方差的先验值。
            - prior_Sigma：状态协方差的先验值。
            - prior_S：观测预测误差协方差的先验值。
        """
        self.seq_len_input = 1 # KNet calculates time-step by time-step

        self.prior_Q = prior_Q.to(self.device)
        self.prior_Sigma = prior_Sigma.to(self.device)
        self.prior_S = prior_S.to(self.device)

        self.m = 4

        # GRU to track Q 预测误差协方差 Q
        self.d_input_Q = self.m * args.in_mult_KNet  # 输入维度，通常是 m 的扩展维度
        self.d_hidden_Q = self.m ** 2  # 隐藏层维度，对应于协方差矩阵的形状
        self.GRU_Q = nn.GRU(self.d_input_Q, self.d_hidden_Q).to(self.device)

        # GRU to track Sigma 状态协方差 Sigma
        self.d_input_Sigma = self.d_hidden_Q + self.m * args.in_mult_KNet
        self.d_hidden_Sigma = self.m ** 2
        self.GRU_Sigma = nn.GRU(self.d_input_Sigma, self.d_hidden_Sigma).to(self.device)
       
        # GRU to track S 观测误差协方差 S
        #self.d_input_S = self.n ** 2 + 2 * (self.n * 2) * args.in_mult_KNet
        self.d_input_S = self.n ** 2 + 2 * self.n  * args.in_mult_KNet
        self.d_hidden_S = (2 * self.n) ** 2
        self.GRU_S = nn.GRU(self.d_input_S, self.d_hidden_S).to(self.device)
        
        # Fully connected 1 用于对 Sigma 的输出进行降维和非线性变换
        self.d_input_FC1 = self.d_hidden_Sigma
        self.d_output_FC1 = self.n ** 2
        self.FC1 = nn.Sequential(
                nn.Linear(self.d_input_FC1, self.d_output_FC1),
                nn.ReLU()).to(self.device)

        # Fully connected 2      对 S 和 Sigma 的输出进行结合
        self.d_input_FC2 = self.d_hidden_S + self.d_hidden_Sigma
        self.d_output_FC2 = 2 * self.n * self.m  
        self.d_hidden_FC2 = self.d_input_FC2 * args.out_mult_KNet
        self.FC2 = nn.Sequential(
                nn.Linear(self.d_input_FC2, self.d_hidden_FC2),
                nn.ReLU(),
                nn.Linear(self.d_hidden_FC2, self.d_output_FC2)
                ).to(self.device)
        
        # Fully connected 3  生成状态协方差矩阵
        self.d_input_FC3 = self.d_hidden_S + self.d_output_FC2
        self.d_output_FC3 = self.m ** 2
        self.FC3 = nn.Sequential(
                nn.Linear(self.d_input_FC3, self.d_output_FC3),
                nn.ReLU()).to(self.device)

        # Fully connected 4 对状态协方差矩阵进行更新
        self.d_input_FC4 = self.d_hidden_Sigma + self.d_output_FC3
        self.d_output_FC4 = self.d_hidden_Sigma
        self.FC4 = nn.Sequential(
                nn.Linear(self.d_input_FC4, self.d_output_FC4),
                nn.ReLU()).to(self.device)
        
        # Fully connected 5 扩展 Kalman 滤波中的状态维度
        self.d_input_FC5 = self.m
        self.d_output_FC5 = self.m * args.in_mult_KNet
        self.FC5 = nn.Sequential(
                nn.Linear(self.d_input_FC5, self.d_output_FC5),
                nn.ReLU()).to(self.device)

        # Fully connected 6 对观测维度进行扩展
        self.d_input_FC6 = self.m
        self.d_output_FC6 = self.m * args.in_mult_KNet
        self.FC6 = nn.Sequential(
                nn.Linear(self.d_input_FC6, self.d_output_FC6),
                nn.ReLU()).to(self.device)

        # Fully connected 7 扩展观测误差协方差矩阵
        #self.d_input_FC7 = 2 * (self.n * 2)
        self.d_input_FC7 = 2 * self.n 
        #self.d_output_FC7 = 2 * (self.n * 2) * args.in_mult_KNet
        self.d_output_FC7 = 2 * self.n  * args.in_mult_KNet
        self.FC7 = nn.Sequential(
                nn.Linear(self.d_input_FC7, self.d_output_FC7),
                nn.ReLU()).to(self.device)

版本说明：改成计算欧拉角画图版

In [ ]:
# 将每个序列分成验证集和训练集（时间序列内切分）
# 这种方法将每个序列的数据根据时间顺序分成两部分：训练集（早期数据）和验证集（后期数据）。

# 特点：
# 时间连续性：保留每个序列的时间相关性。训练集用来训练模型，而验证集用来检查模型的预测能力，验证集永远包含时间更晚的部分。
# 适用场景：常用于时间序列预测任务，例如金融、气象、传感器数据等。这里重要的是时间依赖性，模型需要学会从历史数据中推断未来趋势。
# 对实验结果的影响：
# 时间依赖性保留：确保模型基于过去的数据预测未来，符合实际应用场景中模型的使用方式。
# 训练和测试的分布不同：训练数据和验证数据可能分布不同，因为验证集是序列后期的数据，这可能更具挑战性。序列后期的趋势可能与前期不同，因此更能测试模型的泛化能力。
# 模型的泛化性能更容易评估：这种划分方式与模型实际应用非常接近（预测未来），因此可以更真实地评估模型的性能。
# 2. 将不同的序列分成验证集和训练集（序列间切分）
# 这种方法将整个数据集的不同序列进行划分，一些序列用于训练集，另一些用于验证集。

# 特点：
# 序列独立性：训练集和验证集是完全不同的序列。模型只能从训练集中的序列学习特征，在验证集中的不同序列上评估模型性能。
# 适用场景：常用于任务中不同的序列代表不同的对象、实验条件、或环境（例如不同病人的医疗数据，不同区域的气象数据）。在这些情况下，序列之间的差异可能更重要，训练集和验证集的序列相互独立可以更好地测试模型在新序列上的泛化能力。
# 对实验结果的影响：
# 泛化性能的测试更严格：这种方式严格测试模型的泛化能力，尤其是在训练集和验证集的序列分布不同时。模型不能依赖某个特定序列的模式，而必须学习更加通用的特征。
# 可能更加现实：在一些应用中，未来使用模型时可能会面对完全新的序列（例如新病人或新地区的天气）。因此，这种划分方式更能反映模型在实际应用中遇到新数据时的表现。
# 对比和总结：
# 将每个序列分成训练集和验证集：

# 适合时间序列预测。
# 保留序列内部的时间依赖性，验证模型对时间趋势的预测能力。
# 更适合衡量模型在同一序列上的短期预测能力。
# 将不同序列分成训练集和验证集：

# 适合多个独立序列的任务。
# 强调模型的泛化能力，确保模型在面对新序列时仍能有效预测。
# 更适合测试模型在跨序列上的泛化能力，验证其是否能够处理新的数据模式。
# 选择哪种划分方式取决于实验目标：如果需要模型预测一个序列中的未来数据，第一种方法更合适；如果目标是模型能够应对新的序列，第二种方法会更加严格。

In [ ]:
def loss_fn_1(self, q1, q2):
    """
    计算基于四元数叉乘结果虚部的 L1 范数损失。

    参数:
    q1, q2: 形状为 (N, T, 4) 的四元数张量，N 为批次大小，T 为每批次的四元数数量。

    返回:
    L1 范数损失值
    """
    # 假设输入的四元数已经归一化，否则需要启用下面两行进行归一化
    # q1 = F.normalize(q1, p=2, dim=-1)  # 归一化四元数
    # q2 = F.normalize(q2, p=2, dim=-1)  # 真值已经归一化过

    # 四元数叉乘公式
    # q = [w1, x1, y1, z1] * [w2, x2, y2, z2]
    w1, x1, y1, z1 = q1[..., 0], q1[..., 1], q1[..., 2], q1[..., 3]
    w2, x2, y2, z2 = q2[..., 0], q2[..., 1], q2[..., 2], q2[..., 3]

    # 计算叉乘结果
    w = w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2  # 标量部分
    x = w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2  # 虚部 x
    y = w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2  # 虚部 y
    z = w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2  # 虚部 z

    # 将结果重新组合为四元数
    q_result = torch.stack((w, x, y, z), dim=-1)  # 形状为 (N, T, 4)

    # 提取叉乘结果的虚部
    im_result = q_result[..., 1:]  # 取虚部部分，形状为 (N, T, 3)

    # 计算虚部的 L1 范数
    l1_norm = torch.norm(im_result, p=1, dim=-1)  # L1 范数，形状为 (N, T)

    # 计算最终损失
    loss = 100 * torch.mean(l1_norm)  # 平均损失

    return loss

    def loss_fn_2(self, q1, q2):
        """
        计算两个四元数张量之间的角度差，用于评估模型的性能。

        参数:
        q1, q2: 形状为 (N, T, 4) 的四元数张量，N 为批次大小，T 为每批次的四元数数量。

        返回:
        每个样本的角度误差（以度为单位）
        """
        #q1 = F.normalize(q1, p=2, dim=-1)  # 归一化四元数
        #q2 = F.normalize(q2, p=2, dim=-1)  # 真值已经归一化过
        epsilon = 1e-6
        # 计算四元数内积，按最后一维（4）进行点积，保持 batch_size 和 step 的维度
        dot_product = torch.sum(q1 * q2, dim=-1)  # 点积沿着最后一个维度（即 4 维）

        # 防止由于浮点误差，dot_product 超出 [-1, 1] 范围
        dot_product = torch.clamp(dot_product, min=-1.0 + epsilon, max=1.0 - epsilon)

        # 计算角度（cos^-1）
        theta = 2 * torch.acos(torch.abs(dot_product))  # 得到的是弧度

        # 将弧度转换为度
        theta_deg = torch.rad2deg(theta)  # [batch_size, step]（角度，单位：度）

        # 返回平均角度损失
        return 2 *torch.mean(theta_deg)
    
    def loss_fn_3(self, q1, q2):
        """
        计算两个四元数张量之间的角度差，用于评估模型的性能。

        参数:
        q1, q2: 形状为 (N, T, 4) 的四元数张量，N 为批次大小，T 为每批次的四元数数量。

        返回:
        1-|q1点乘q2|
        """
        #q1 = F.normalize(q1, p=2, dim=-1)  # 归一化四元数
        #q2 = F.normalize(q2, p=2, dim=-1)  # 真值已经归一化过

        # 计算四元数内积，按最后一维（4）进行点积，保持 batch_size 和 step 的维度
        dot_product = torch.sum(q1 * q2, dim=-1)  # 点积沿着最后一个维度（即 4 维）

        Loss = 1 - torch.abs(dot_product)
        mean_loss = torch.mean(Loss)

        # 返回平均角度损失
        return 100 * mean_loss